In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df= pd.read_csv('../data/router_telemetry_dataset.csv')
df.head()

,router_id,vendor,router_model,ip_address,slot_id,cpu_percent,memory_percent,heap_percent,temperature_c,fan_speed_rpm,power_draw_w,interface_error_count,packet_drop_rate,latency_ms,uptime_hours,alarm_count,card_status,link_status,isolate_required
0,RTR-0080,Cisco,CRS-1,10.156.140.109,slot-6,44.36,43.99,40.66,45.0,6014,620.9,254,6.436,131.31,17622.0,4,ok,up,1
1,RTR-2847,Ciena,3924,10.220.26.170,slot-7,43.39,37.57,38.84,60.1,5388,692.3,250,5.871,120.15,13907.4,4,ok,up,1
2,RTR-2484,Cisco,CRS-1,10.202.3.241,slot-1,66.11,15.89,23.07,46.7,6683,248.7,152,3.145,58.89,31815.3,2,ok,up,0
3,RTR-0524,Juniper,MX480,10.240.4.177,slot-6,47.10,48.46,39.19,41.2,5968,553.7,267,5.402,110.12,7314.8,3,warning,up,1
4,RTR-0855,Cisco,ASR9001,10.57.179.123,slot-2,21.61,41.84,28.17,56.8,7394,398.0,118,3.490,59.66,49896.6,1,ok,up,0


In [2]:
print(df.duplicated().sum())


90


In [3]:
print(df.isnull().sum())

router_id                0
vendor                   0
router_model             0
ip_address               0
slot_id                  0
cpu_percent              0
memory_percent           0
heap_percent             0
temperature_c            0
fan_speed_rpm            0
power_draw_w             0
interface_error_count    0
packet_drop_rate         0
latency_ms               0
uptime_hours             0
alarm_count              0
card_status              0
link_status              0
isolate_required         0
dtype: int64


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3090 entries, 0 to 3089
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   router_id              3090 non-null   object 
 1   vendor                 3090 non-null   object 
 2   router_model           3090 non-null   object 
 3   ip_address             3090 non-null   object 
 4   slot_id                3090 non-null   object 
 5   cpu_percent            3090 non-null   float64
 6   memory_percent         3090 non-null   float64
 7   heap_percent           3090 non-null   float64
 8   temperature_c          3090 non-null   float64
 9   fan_speed_rpm          3090 non-null   int64  
 10  power_draw_w           3090 non-null   float64
 11  interface_error_count  3090 non-null   int64  
 12  packet_drop_rate       3090 non-null   float64
 13  latency_ms             3090 non-null   float64
 14  uptime_hours           3090 non-null   float64
 15  alar

In [5]:
df=df.drop_duplicates()

In [6]:
df.shape

(3000, 19)

In [7]:
#enforce data type of columns
obj_cols=df.select_dtypes(include=['object']).columns
for col in obj_cols:
    df[col] = df[col].astype('object')


In [8]:
df = df.drop(axis=1,columns=['router_id','ip_address'])

In [9]:
df[['slot_id','router_model','vendor','card_status','link_status']].nunique(axis=0)

slot_id          8
router_model    20
vendor           5
card_status      4
link_status      3
dtype: int64

In [10]:
#checking for outliers using IQR
num_col_list=df.select_dtypes(include=['number']).columns
outlier_summary = []

for col in num_col_list:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    outlier_summary.append({
        'column': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': len(outliers)
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
print(outlier_summary_df.sort_values('outlier_count', ascending=False))

                   column           Q1           Q3         IQR  lower_bound  \
11       isolate_required      0.00000      0.00000      0.0000      0.00000   
8              latency_ms     58.79250    128.54500     69.7525    -45.83625   
10            alarm_count      2.00000      4.00000      2.0000     -1.00000   
1          memory_percent     32.26750     49.60500     17.3375      6.26125   
0             cpu_percent     27.98500     46.61500     18.6300      0.04000   
7        packet_drop_rate      2.99825      6.40625      3.4080     -2.11375   
2            heap_percent     25.86750     42.80000     16.9325      0.46875   
3           temperature_c     42.20000     58.92500     16.7250     17.11250   
5            power_draw_w    399.90000    599.87500    199.9750     99.93750   
6   interface_error_count    119.00000    257.00000    138.0000    -88.00000   
4           fan_speed_rpm   5937.00000   7248.00000   1311.0000   3970.50000   
9            uptime_hours  12822.37500  

In [11]:
#removing outliers rows which have more than half of the numeric columns as outliers
num_col_list=df.select_dtypes(include=['number']).drop(columns=['isolate_required']).columns
numeric_cols = [
    'cpu_percent', 'memory_percent', 'heap_percent', 'temperature_c',
    'fan_speed_rpm', 'power_draw_w', 'interface_error_count',
    'packet_drop_rate', 'latency_ms', 'uptime_hours', 'alarm_count'
]

outlier_flags = pd.DataFrame(index=df.index)

for col in num_col_list:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outlier_flags[col] = ((df[col] < lower) | (df[col] > upper)).astype(int)

df['outlier_count'] = outlier_flags.sum(axis=1)
df['outlier_ratio'] = df['outlier_count'] / len(numeric_cols)

threshold = len(numeric_cols) / 2
rows_to_remove = df[df['outlier_count'] >= threshold]
df_cleaned = df[df['outlier_count'] < threshold].copy()

print("Total rows:", len(df))
print("Rows to remove:", len(rows_to_remove))
print("Rows left:", len(df_cleaned))
df = df_cleaned.drop(columns=['outlier_count', 'outlier_ratio'])
df.head()


Total rows: 3000
Rows to remove: 5
Rows left: 2995


,vendor,router_model,slot_id,cpu_percent,memory_percent,heap_percent,temperature_c,fan_speed_rpm,power_draw_w,interface_error_count,packet_drop_rate,latency_ms,uptime_hours,alarm_count,card_status,link_status,isolate_required
0,Cisco,CRS-1,slot-6,44.36,43.99,40.66,45.0,6014,620.9,254,6.436,131.31,17622.0,4,ok,up,1
1,Ciena,3924,slot-7,43.39,37.57,38.84,60.1,5388,692.3,250,5.871,120.15,13907.4,4,ok,up,1
2,Cisco,CRS-1,slot-1,66.11,15.89,23.07,46.7,6683,248.7,152,3.145,58.89,31815.3,2,ok,up,0
3,Juniper,MX480,slot-6,47.10,48.46,39.19,41.2,5968,553.7,267,5.402,110.12,7314.8,3,warning,up,1
4,Cisco,ASR9001,slot-2,21.61,41.84,28.17,56.8,7394,398.0,118,3.490,59.66,49896.6,1,ok,up,0


In [12]:
obj_cols = df.select_dtypes(include=['object', 'string']).columns
df[obj_cols] = df[obj_cols].apply(lambda x: x.str.strip())
df[obj_cols].head()

,vendor,router_model,slot_id,card_status,link_status
0,Cisco,CRS-1,slot-6,ok,up
1,Ciena,3924,slot-7,ok,up
2,Cisco,CRS-1,slot-1,ok,up
3,Juniper,MX480,slot-6,warning,up
4,Cisco,ASR9001,slot-2,ok,up


In [13]:
df.head()

,vendor,router_model,slot_id,cpu_percent,memory_percent,heap_percent,temperature_c,fan_speed_rpm,power_draw_w,interface_error_count,packet_drop_rate,latency_ms,uptime_hours,alarm_count,card_status,link_status,isolate_required
0,Cisco,CRS-1,slot-6,44.36,43.99,40.66,45.0,6014,620.9,254,6.436,131.31,17622.0,4,ok,up,1
1,Ciena,3924,slot-7,43.39,37.57,38.84,60.1,5388,692.3,250,5.871,120.15,13907.4,4,ok,up,1
2,Cisco,CRS-1,slot-1,66.11,15.89,23.07,46.7,6683,248.7,152,3.145,58.89,31815.3,2,ok,up,0
3,Juniper,MX480,slot-6,47.10,48.46,39.19,41.2,5968,553.7,267,5.402,110.12,7314.8,3,warning,up,1
4,Cisco,ASR9001,slot-2,21.61,41.84,28.17,56.8,7394,398.0,118,3.490,59.66,49896.6,1,ok,up,0


In [14]:
for col in df.select_dtypes(include=['object']).columns:
    print(f"Value counts for {col}:")
    print(df[col].value_counts())


Value counts for vendor:
vendor
Cisco      765
Juniper    606
Huawei     601
Nokia      574
Ciena      449
Name: count, dtype: int64
Value counts for router_model:
router_model
ASR9001          204
CRS-1            192
ASR9006          189
NCS5500          180
PTX5000          176
1830 PSS         172
NE40E-X3         156
NE40E-X8         151
MX960            150
ATN 910C         149
7210-SAS         147
CX600-X8         145
MX480            143
7750-SR12        141
QFX10008         137
6500-T32         117
8180             115
7750-SR7         114
Waveserver Ai    112
3924             105
Name: count, dtype: int64
Value counts for slot_id:
slot_id
slot-5    407
slot-7    398
slot-2    388
slot-3    373
slot-6    372
slot-1    358
slot-4    358
slot-0    341
Name: count, dtype: int64
Value counts for card_status:
card_status
ok          2468
warning      425
degraded      96
failed         6
Name: count, dtype: int64
Value counts for link_status:
link_status
up          2569
unstable  

In [15]:
df.to_csv('../data/cleaned_router_telemetry_dataset.csv', index=False)